# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aun-Mehdi117/-Flyrank-ML-Internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

**Lane 2 — Refresh / Content Opportunity Scoring.** The final deliverable notebook: turns the
Week-4 baseline rule and the Week-5/6 model (`w05_model.ipynb`, honest client-grouped split
confirmed in `w06_validation_audit.ipynb`) into one ranked, reason-coded action queue, states
who it's for and where it stops being valid, and exports the files the capstone paper builds on.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

Score the held-out test split (never the rows the model trained on) with the winning Logistic
Regression pipeline from ML-08, recompute the Week-4 baseline rule on the same rows, and
combine both into three reason codes rather than trusting either signal alone.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

NUMERIC_FEATURES = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'impressions_90d', 'clicks_90d', 'sessions_90d', 'ai_sessions_90d',
    'days_with_impressions', 'days_with_sessions', 'content_age_days',
    'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate',
    'scroll_rate', 'ai_traffic_pct',
]
CATEGORICAL_FEATURES = [
    'competition_level', 'content_type', 'main_intent', 'age_tier',
    'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier',
]
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
for c in NUMERIC_FEATURES:
    df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0)
for c in CATEGORICAL_FEATURES:
    df[c] = df[c].fillna('unknown').astype(str)
for c in ['impressions_90d', 'clicks_90d', 'sessions_90d', 'ai_sessions_90d']:
    df[f'log_{c}'] = np.log1p(df[c])
NUM_FINAL = [c for c in NUMERIC_FEATURES if c not in
             ['impressions_90d', 'clicks_90d', 'sessions_90d', 'ai_sessions_90d']] + \
            ['log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d']

X = df[NUM_FINAL + CATEGORICAL_FEATURES]
y = df['is_declining_label']
groups = df['client_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

pre = ColumnTransformer([
    ('num', StandardScaler(), NUM_FINAL),
    ('cat', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL_FEATURES),
])
pipe = Pipeline([('pre', pre), ('clf', LogisticRegression(max_iter=1000, random_state=42))])
pipe.fit(X.iloc[train_idx], y.iloc[train_idx])
test_proba = pipe.predict_proba(X.iloc[test_idx])[:, 1]
print(f'Scored {len(test_idx):,} held-out rows (never seen during training).')

Scored 6,163 held-out rows (never seen during training).


In [2]:
# Week-4 baseline rule, recomputed on the same rows (same formula as w04_baseline_score.ipynb
# and w05_model.ipynb -- never re-derived differently between notebooks).
ctr_benchmark = {'top_3': 0.334128, 'page_1': 0.354760, 'striking': 0.255782,
                 'page_3_5': 0.142359, 'deep': 0.055415}
stale        = (df['days_since_last_update'] >= 90).astype(int)
visible      = (df['impressions_90d'] >= 100).astype(int)
position_ok  = df['position_tier'].isin(['top_3', 'page_1', 'striking', 'page_3_5']).astype(int)
benchmark    = df['position_tier'].map(ctr_benchmark).fillna(0)
ctr_gap      = (benchmark - df['ctr']).clip(lower=0)
baseline_score_full = (stale * visible * position_ok * ctr_gap * np.log1p(df['impressions_90d'])).values

test_df = df.iloc[test_idx].copy()
test_df['model_proba']      = test_proba
test_df['baseline_score']   = baseline_score_full[test_idx]
test_df['baseline_flagged'] = test_df['baseline_score'] > 0
test_df['actual']           = y.iloc[test_idx].values

# Model flag = top 10% of predicted risk on this held-out split.
thresh = np.quantile(test_proba, 0.90)
test_df['model_flagged'] = test_df['model_proba'] >= thresh
print(f'Model top-10% probability threshold: {thresh:.3f}')

Model top-10% probability threshold: 0.790


In [3]:
def reason(row):
    if row['model_flagged'] and row['baseline_flagged']:
        return 'baseline_and_model_agree'
    elif row['model_flagged']:
        return 'model_only_high_risk'
    elif row['baseline_flagged']:
        return 'baseline_only_ctr_gap'
    return 'not_flagged'

test_df['reason_code'] = test_df.apply(reason, axis=1)

REASON_TEXT = {
    'baseline_and_model_agree': 'Stale, visible, underperforming its position AND the model '
                                 'independently ranks it high risk -- two different signals pointing the same way.',
    'model_only_high_risk':     'Model ranks this high risk even though it does not trip the '
                                 'stale/CTR-gap rule -- often large pages with 0% CTR (see the review note below).',
    'baseline_only_ctr_gap':    'Trips the stale + CTR-gap rule, but the model does not rank it '
                                 'unusually risky -- lower priority than the two buckets above.',
}

quality = (test_df[test_df['reason_code'] != 'not_flagged']
           .groupby('reason_code')
           .agg(n=('content_id', 'count'), observed_decline_rate=('actual', 'mean')))
quality['reason_text'] = quality.index.map(REASON_TEXT)
print(f"Base rate for reference (test split): {test_df['actual'].mean():.3f}\n")
print(quality[['n', 'observed_decline_rate', 'reason_text']].to_string())

Base rate for reference (test split): 0.511

                            n  observed_decline_rate                                                                                                                                   reason_text
reason_code                                                                                                                                                                                       
baseline_and_model_agree  152               0.611842   Stale, visible, underperforming its position AND the model independently ranks it high risk -- two different signals pointing the same way.
baseline_only_ctr_gap     350               0.514286                  Trips the stale + CTR-gap rule, but the model does not rank it unusually risky -- lower priority than the two buckets above.
model_only_high_risk      465               0.677419  Model ranks this high risk even though it does not trip the stale/CTR-gap rule -- often large pages with 0% CTR (see the 

In [4]:
queue = (test_df[test_df['reason_code'] != 'not_flagged']
         .sort_values('model_proba', ascending=False)
         .reset_index(drop=True))

print(f'Ranked queue: {len(queue):,} pages (of {len(test_df):,} scored)')
cols = ['content_id', 'client_id', 'model_proba', 'baseline_score', 'reason_code',
        'ctr', 'impressions_90d', 'position_tier', 'days_since_last_update']
queue[cols].head(10)

Ranked queue: 967 pages (of 6,163 scored)


**Reading the queue in words a human trusts:** `baseline_and_model_agree` pages are the
clearest call — 61.2% of them actually declined (vs. a 51.1% base rate), and two independent
signals point the same way. `model_only_high_risk` pages score *even higher* in practice
(67.7% observed decline rate) but wouldn't have been caught by the Week-4 rule alone — worth
digging into why in section 3. `baseline_only_ctr_gap` pages sit at 51.4%, barely above the
base rate — the rule flags them, but the model doesn't see anything unusual, so these are
lower priority than the label "flagged" alone would suggest.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [5]:
print('Intended use: a content ops lead or SEO strategist deciding which pages to review')
print('first in a monthly refresh cycle, out of a portfolio too large to review page-by-page.')
print()
print('Where it stops being valid:')
print(f'  - Trained on {df["client_id"].nunique()} clients, {len(df):,} rows, one 90-day')
print('    local snapshot -- not the full FlyRank warehouse. Re-validate before trusting this')
print('    queue on a client industry or content type not represented here.')
print('  - The label (is_declining) is a 30d-vs-prev-30d impression trend, not a business')
print('    outcome -- it does not know about seasonality, algorithm updates, or planned')
print('    campaigns that would move impressions for reasons unrelated to content quality.')
print('  - Cross-sectional, not causal: this ranks pages by observed association with decline,')
print('    it does not claim that refreshing a flagged page WILL fix it (see w06 section 4).')
print('  - Grouped-split AUC was 0.616 -- meaningfully better than chance, far from perfect.')
print('    Expect real false positives and false negatives at the volumes seen in ML-08.')

Intended use: a content ops lead or SEO strategist deciding which pages to review
first in a monthly refresh cycle, out of a portfolio too large to review page-by-page.

Where it stops being valid:
  - Trained on 32 clients, 30,000 rows, one 90-day
    local snapshot -- not the full FlyRank warehouse. Re-validate before trusting this
    queue on a client industry or content type not represented here.
  - The label (is_declining) is a 30d-vs-prev-30d impression trend, not a business
    outcome -- it does not know about seasonality, algorithm updates, or planned
    campaigns that would move impressions for reasons unrelated to content quality.
  - Cross-sectional, not causal: this ranks pages by observed association with decline,
    it does not claim that refreshing a flagged page WILL fix it (see w06 section 4).
  - Grouped-split AUC was 0.616 -- meaningfully better than chance, far from perfect.
    Expect real false positives and false negatives at the volumes seen in ML-08.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [6]:
# The concrete review trigger: ML-08 found false positives were dominated by large, established
# pages sitting at exactly 0% CTR. Confirm how much of THIS queue's model_only bucket matches
# that same pattern before trusting it blindly.
model_only = test_df[test_df['reason_code'] == 'model_only_high_risk']
zero_ctr = (model_only['ctr'] == 0).sum()
print(f'model_only_high_risk pages at exactly 0% CTR: {zero_ctr} of {len(model_only)} '
      f'({zero_ctr/len(model_only):.0%})')
print()
print('That is the single most important human-review trigger this queue produces: a 0% CTR on')
print('a page with real impression volume is AS LIKELY to be a tracking/tagging problem as a')
print('genuine content problem, and no field in this dataset can tell the two apart.')

model_only_high_risk pages at exactly 0% CTR: 239 of 465 (51%)

That is the single most important human-review trigger this queue produces: a 0% CTR on
a page with real impression volume is AS LIKELY to be a tracking/tagging problem as a
genuine content problem, and no field in this dataset can tell the two apart.


In [7]:
print('Before acting on any page in this queue, a human must check:')
print('  1. Does the page still exist and resolve correctly (not a broken/redirected URL)?')
print('  2. For 0% CTR pages: is analytics/search-console tracking actually firing on it?')
print('  3. Is the decline portfolio-wide for this client (a tracking or seasonal issue), or')
print('     specific to this page?')
print('  4. Does model_proba disagree sharply with baseline_score in a way that is unexplained')
print('     by (1)-(3) -- if so, treat it as "needs investigation", not "needs refresh".')
print()
print('NO-GO list -- never automate:')
print('  - Never auto-publish or auto-edit content based on this score.')
print('  - Never merge, prune, or de-index a page from this score alone -- those are one-way')
print('    doors and this model was never validated against that outcome.')
print('  - Never treat model_proba as a probability of TRUTH -- it is a rank, not a guarantee')
print('    (AUC 0.616 means real mistakes at these volumes).')
print('  - Never act on a reason-code bucket below the n=50 sample floor without flagging that')
print('    explicitly to whoever reviews it.')

Before acting on any page in this queue, a human must check:
  1. Does the page still exist and resolve correctly (not a broken/redirected URL)?
  2. For 0% CTR pages: is analytics/search-console tracking actually firing on it?
  3. Is the decline portfolio-wide for this client (a tracking or seasonal issue), or
     specific to this page?
  4. Does model_proba disagree sharply with baseline_score in a way that is unexplained
     by (1)-(3) -- if so, treat it as "needs investigation", not "needs refresh".

NO-GO list -- never automate:
  - Never auto-publish or auto-edit content based on this score.
  - Never merge, prune, or de-index a page from this score alone -- those are one-way
    doors and this model was never validated against that outcome.
  - Never treat model_proba as a probability of TRUTH -- it is a rank, not a guarantee
    (AUC 0.616 means real mistakes at these volumes).
  - Never act on a reason-code bucket below the n=50 sample floor without flagging that
    explic

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [8]:
print('Monitoring plan:')
print('  - Track precision@K on each new month\'s data using the SAME grouped-by-client split')
print('    logic as w06 -- a silent drop below the levels measured here (precision@20 ~0.70,')
print('    precision@50 ~0.72) is the primary retrain trigger.')
print('  - Track the base rate of is_declining month over month -- a large shift means the')
print('    portfolio composition changed and the model needs re-validation, not just re-scoring.')
print('  - Track how often new clients enter the queue with little history -- a client the model')
print('    has never seen in training is exactly the cold-start case the grouped split exists to')
print('    simulate, and it is worth watching in production for real, not just in a test split.')
print('  - Track the model_only_high_risk zero-CTR rate over time -- if it climbs, that is a')
print('    signal the model is increasingly keying off a tracking artifact rather than content')
print('    quality, and the feature set needs a second look before the next retrain.')
print('  - Retrain cadence: monthly, using the newest full month as the new held-out test split')
print('    and re-running the leakage checklist from w06 fresh each time -- a passing check last')
print('    month does not carry forward automatically.')

Monitoring plan:
  - Track precision@K on each new month's data using the SAME grouped-by-client split
    logic as w06 -- a silent drop below the levels measured here (precision@20 ~0.70,
    precision@50 ~0.72) is the primary retrain trigger.
  - Track the base rate of is_declining month over month -- a large shift means the
    portfolio composition changed and the model needs re-validation, not just re-scoring.
  - Track how often new clients enter the queue with little history -- a client the model
    has never seen in training is exactly the cold-start case the grouped split exists to
    simulate, and it is worth watching in production for real, not just in a test split.
  - Track the model_only_high_risk zero-CTR rate over time -- if it climbs, that is a
    signal the model is increasingly keying off a tracking artifact rather than content
    quality, and the feature set needs a second look before the next retrain.
  - Retrain cadence: monthly, using the newest full month as

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to `work/outputs/` — your paper builds on
these files.*

In [9]:
import os
os.makedirs('../outputs', exist_ok=True)

export_cols = ['content_id', 'client_id', 'model_proba', 'baseline_score', 'reason_code',
               'ctr', 'impressions_90d', 'position_tier', 'days_since_last_update', 'actual']
queue[export_cols].to_csv('../outputs/ranked_action_queue.csv', index=False)

quality_export = quality[['n', 'observed_decline_rate']].reset_index()
quality_export.to_csv('../outputs/reason_code_quality.csv', index=False)

print('Exported:')
print(f'  work/outputs/ranked_action_queue.csv    ({len(queue):,} rows)')
print(f'  work/outputs/reason_code_quality.csv    ({len(quality_export)} rows)')

Exported:
  work/outputs/ranked_action_queue.csv    (967 rows)
  work/outputs/reason_code_quality.csv    (3 rows)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.